In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

# Load the CSV file
df = pd.read_csv(r"C:\Users\mallu\Downloads\owid-co2-data.csv")

print("First 10 Rows:")
print(df.head(10))

print("\nColumn Names:")
print(df.columns.tolist())

print("\nData Types:")
print(df.dtypes)

print("\nShape (rows, columns):", df.shape)

# Descrption of key columns
-**co2** : carbon dioxide emmision by the country.
-**co2_per_capita** :  total co2 emission of country divided by to total number of people of a country.
-**methane** : methane gas emitted by each country.
-**nitrous_oxide** : nitrous oxide emissions produced by the country.
-**total_ghg** : total greenhouse gas emissions including co2, methane,nitrous oxide & other greenhouse gas.
-**country** :the name of the country for which green house gas emision is recorded.
-**year**: the year in which the emission data is collected.

In [ ]:
# Null values per column as percentage of total rows
null_percentage = (df.isnull().sum() / len(df)) * 100
null_percentage = null_percentage.sort_values(ascending=False)

print("Null Values per Column (% of total rows):")
print(null_percentage.round(2))

In [ ]:
# Completeness per row (% of non-null columns)
df['completeness'] = df.notnull().mean(axis=1) * 100

country_completeness = (
    df.groupby('country')['completeness']
    .mean()
    .sort_values(ascending=False)
)

print("Top 10 Countries by Data Completeness:")
print(country_completeness.head(10).round(2))

year_completeness = (
    df.groupby('year')['completeness']
    .mean()
    .sort_values(ascending=False)
)

print("\nTop 10 Years by Data Completeness:")
print(year_completeness.head(10).round(2))

In [ ]:
# Aggregate / non-sovereign entities to exclude (regions, income groups, trade blocs, etc.)
aggregates_to_exclude = [
    'World', 'Asia', 'Europe', 'Africa', 'North America', 'South America', 'Oceania',
    'European Union (27)', 'European Union (28)',
    'High-income countries', 'Low-income countries',
    'Upper-middle-income countries', 'Lower-middle-income countries',
    'Asia (excl. China and India)', 'Europe (excl. EU-27)', 'Europe (excl. EU-28)',
    'North America (excl. USA)',
    'International transport', 'International aviation', 'International shipping',
    'Kuwaiti Oil Fires', 'Kuwaiti Oil Fires (GCP)',
    'OECD (GCP)', 'OECD (Jones et al.)', 'Non-OECD (GCP)', 'G20', 'G7',
    'Africa (GCP)', 'Asia (GCP)', 'Europe (GCP)', 'Middle East (GCP)',
    'North America (GCP)', 'Oceania (GCP)', 'South America (GCP)',
    'Central America (GCP)', 'Ryukyu Islands (GCP)'
]

# Apply filters: year >= 1990, exclude aggregates (keep ALL real countries)
df_filtered = df[
    (df['year'] >= 1990) &
    (~df['country'].isin(aggregates_to_exclude))
].copy()

print("Filtered dataset shape:", df_filtered.shape)
print("Number of countries included:", df_filtered['country'].nunique())
print("Year range:", df_filtered['year'].min(), "-", df_filtered['year'].max())
df_filtered.head(10)

Filtering Decisions and Justification

The original OWID dataset contains 50,411 rows across 254 entities, covering the years 1750–2024. Before analysis, two filters were applied to create a clean country-level dataset.

1. Keep data from 1990 onwards

Only records from 1990–2024 were kept because earlier data has many missing values for important variables like total_ghg, methane, nitrous_oxide, and gdp. In addition, 1990 is the standard baseline year used in many international climate agreements, making it a meaningful starting point for the analysis.

2. Remove non-country entities

The dataset includes entries such as World, Asia, Europe, G20, OECD, and other regional or economic groups along with individual countries. These were removed because they represent combined totals rather than actual countries. Keeping them would lead to double-counting and produce misleading results in country-level analysis. A curated list of these aggregate entities was used to filter them out.

Final Dataset

After applying these filters, the dataset contains 7,735 rows covering 221 sovereign countries from 1990–2024. This provides a clean and reliable dataset for further analysis, visualization, and feature engineering.

In [ ]:
# Global CO2 emissions computed from the filtered sovereign-country dataset
global_co2 = df_filtered.groupby('year')['co2'].sum().reset_index()

plt.figure(figsize=(10, 5))
plt.plot(global_co2['year'], global_co2['co2'], color='darkred', linewidth=2)
plt.title('Global CO\u2082 Emissions (1990 - Latest Year)')
plt.xlabel('Year')
plt.ylabel('CO\u2082 Emissions (million tonnes)')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**Summary — Global CO₂ Emissions Trend (1990–2024)**

Global CO₂ emissions increases 22,184 MtCO₂ in 1990 to about 37,406 MtCO₂ in 2024, an increase of approximately 69% over the 34-year period. The trend is upward and accelerates noticeably through the 2000s, reflecting rapid industrialisation in large developing economies. A clear dip is visible in 2020 (≈34,319 MtCO₂), corresponding to the global COVID-19 pandemic and the associated drop in industrial activity
. Emissions started increasing again just after 2020 & reached a all time by 2024.

*Note: these figures were computed from OWID's official 'World' row. Since this chart now sums the filtered sovereign countries instead, re-check these exact numbers against your new output and update if they've shifted slightly.*

In [ ]:
# Top 5 CO2 emitting countries, computed from the filtered dataset
top5 = ['China', 'United States', 'India', 'Russia', 'Japan']
colors_map = {
    'China':         '#E63946',
    'United States': '#457B9D',
    'India':         '#F4A261',
    'Russia':        '#6A4C93',
    'Japan':         '#2A9D8F',
}

df_top5 = df_filtered[df_filtered['country'].isin(top5)][['country', 'year', 'co2']].dropna()
df_pivot = df_top5.pivot(index='year', columns='country', values='co2')

fig, ax = plt.subplots(figsize=(13, 7))

for country in top5:
    if country in df_pivot.columns:
        ax.plot(df_pivot.index, df_pivot[country],
                label=country, color=colors_map[country],
                linewidth=2.5, marker='o', markersize=3, markevery=5)
        last_yr  = df_pivot[country].dropna().index[-1]
        last_val = df_pivot[country].dropna().iloc[-1]
        ax.annotate(country,
                    xy=(last_yr, last_val), xytext=(6, 0),
                    textcoords='offset points', va='center',
                    fontsize=9, color=colors_map[country], fontweight='bold')

ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.set_xlim(df_pivot.index.min(), df_pivot.index.max() + 6)
ax.set_title('CO\u2082 Emission Trends — Top 5 Emitting Countries (1990–Latest)',
             fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Year', fontsize=11)
ax.set_ylabel('CO\u2082 Emissions (million tonnes)', fontsize=11)
ax.grid(axis='y', linestyle='--', alpha=0.35)
ax.spines[['top', 'right']].set_visible(False)
ax.legend(loc='upper left', fontsize=10, framealpha=0.85)
plt.tight_layout()
plt.show()

**Summary — Top 5 Emitters Comparison (China, USA, India, Russia, Japan)**

In 1990, the United States was the clearly the most co2 emission contry roughly around 5000mtco2, almost double to china & russia, china co2 emission takes rapid growth around 2002 & it went on increaing rapidely leaving all countries behind, it is now leadig c02 emission country around the world. while usa maintained teir co2 emission the years. still it is the second highest co2 emission country around the world.india is also shows increasing in their co2 emission now at third position in the world, whilw russia & japan shows decrease in their co2 emission thats a great sign.

In [ ]:
# Global GHG share by gas type per decade, computed from the filtered dataset
ghg_world = df_filtered.groupby('year')[['co2', 'methane', 'nitrous_oxide']].sum().reset_index()

def decade_label(y):
    if 1990 <= y <= 1999: return '1990s'
    elif 2000 <= y <= 2009: return '2000s'
    elif 2010 <= y <= 2019: return '2010s'
    elif y >= 2020:         return '2020s'
    else:                   return None

ghg_world['decade'] = ghg_world['year'].apply(decade_label)
ghg_world = ghg_world[ghg_world['decade'].notna()]

decade_avg = (
    ghg_world.groupby('decade')[['co2', 'methane', 'nitrous_oxide']]
    .mean()
    .reindex(['1990s', '2000s', '2010s', '2020s'])
)

decade_pct = decade_avg.div(decade_avg.sum(axis=1), axis=0) * 100

gas_labels = ['CO\u2082', 'Methane (CH\u2084)', 'Nitrous Oxide (N\u2082O)']
gas_colors = ['#E63946', '#F4A261', '#457B9D']

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

x = np.arange(len(decade_pct))
bottom = np.zeros(len(decade_pct))

for col, label, color in zip(decade_pct.columns, gas_labels, gas_colors):
    axes[0].bar(x, decade_pct[col], bottom=bottom,
                label=label, color=color, alpha=0.88, width=0.5, edgecolor='white')
    for i, (val, bot) in enumerate(zip(decade_pct[col], bottom)):
        if val > 3:
            axes[0].text(i, bot + val / 2, f'{val:.1f}%',
                         ha='center', va='center', fontsize=9,
                         color='white', fontweight='bold')
    bottom += decade_pct[col].values

axes[0].set_xticks(x)
axes[0].set_xticklabels(decade_pct.index, fontsize=11)
axes[0].set_ylabel('Share of Total GHG (%)', fontsize=11)
axes[0].set_title('GHG Share by Gas Type per Decade\n(% of total, global average)',
                  fontsize=12, fontweight='bold')
axes[0].set_ylim(0, 105)
axes[0].legend(loc='upper right', fontsize=9, framealpha=0.85)
axes[0].spines[['top', 'right']].set_visible(False)
axes[0].grid(axis='y', linestyle='--', alpha=0.3)

bottom2 = np.zeros(len(decade_avg))
for col, label, color in zip(decade_avg.columns, gas_labels, gas_colors):
    axes[1].bar(x, decade_avg[col], bottom=bottom2,
                label=label, color=color, alpha=0.88, width=0.5, edgecolor='white')
    bottom2 += decade_avg[col].values

axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:,.0f}'))
axes[1].set_xticks(x)
axes[1].set_xticklabels(decade_avg.index, fontsize=11)
axes[1].set_ylabel('Avg Annual Emissions (million tonnes CO\u2082-eq)', fontsize=11)
axes[1].set_title('GHG Absolute Emissions by Gas Type per Decade\n(global average per year)',
                  fontsize=12, fontweight='bold')
axes[1].legend(loc='upper left', fontsize=9, framealpha=0.85)
axes[1].spines[['top', 'right']].set_visible(False)
axes[1].grid(axis='y', linestyle='--', alpha=0.3)

plt.suptitle('Global Greenhouse Gas Emissions by Type — Decade Analysis',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

**Summary — Share of Global GHG by Gas Type per Decade**

CO₂ has consistently been the dominant greenhouse gas across all four decades, accounting for
roughly 68–73% of combined CO₂/methane/N₂O emissions throughout the period. Its share rose
steadily from about 68.5% in the 1990s to around 73% by the 2010s, before levelling off in the
2020s. Methane's share moved in the opposite direction, declining from roughly 23.8% in the
1990s to about 20.4–20.6% in the 2010s and 2020s, suggesting CO₂-generating activity (energy
and industry) has grown faster than methane-generating activity (agriculture, waste) over this
period. Nitrous oxide remains the smallest contributor throughout, shrinking modestly from
about 7.7% to 6.6% of the total. Overall, the composition has shifted modestly but consistently
toward CO₂ dominance over the three and a half decades covered.

*Note: recomputed from summed filtered-country data rather than OWID's 'World' row — percentages should be very close to before; re-check and update if any shifted.*

*****Week 2: Feature Engineering*****

In [ ]:
# Reuse Week 1's filtered dataset as the single source of truth for Week 2
df_clean = df_filtered.copy().sort_values(['country', 'year']).reset_index(drop=True)

print(f"Filtered dataset shape : {df_clean.shape}")
print(f"Unique sovereign nations: {df_clean['country'].nunique()}")
print(f"Year range             : {df_clean['year'].min()} \u2013 {df_clean['year'].max()}")
df_clean[['country', 'year', 'co2', 'co2_per_capita', 'total_ghg', 'gdp', 'population']].head(5)

In [ ]:
# ── 2.1 Time-Based Features ──────────────────────────────────────────────

# Decade label: floor-divide year by 10, multiply back, append 's'
df_clean['decade'] = (df_clean['year'] // 10 * 10).astype(str) + 's'

# Simple numeric time index anchored at 1990
df_clean['years_since_1990'] = df_clean['year'] - 1990

# 5-year rolling average of CO2 per country (window=5, min_periods=1 avoids early NaNs)
df_clean['co2_5yr_rolling_mean'] = (
    df_clean
    .groupby('country')['co2']
    .transform(lambda series: series.rolling(window=5, min_periods=1).mean())
)

china_sample = (
    df_clean[df_clean['country'] == 'China']
    [['country', 'year', 'decade', 'years_since_1990', 'co2', 'co2_5yr_rolling_mean']]
    .head(10)
)
print("Sample output for China (first 10 rows after 1990):")
print(china_sample.to_string(index=False))

**Lag features** capture a country's own recent emission history as predictors — `co2_lag1`, `co2_lag2`, `co2_lag3` are the CO₂ values from 1, 2, and 3 years prior for that same country. They're useful because emissions are highly autocorrelated (this year's value is strongly related to last year's), so lags let a model like Linear Regression or Random Forest learn from a country's own trajectory rather than only from static features like GDP or population.

In [ ]:
# ── 2.2 Lag Features ────────────────────────────────────────────────

df_clean['co2_lag1'] = df_clean.groupby('country')['co2'].shift(1)
df_clean['co2_lag2'] = df_clean.groupby('country')['co2'].shift(2)
df_clean['co2_lag3'] = df_clean.groupby('country')['co2'].shift(3)

usa_sample = (
    df_clean[df_clean['country'] == 'United States']
    [['country', 'year', 'co2', 'co2_lag1', 'co2_lag2', 'co2_lag3']]
    .head(6)
)
print("Lag feature verification for United States:")
print(usa_sample.to_string(index=False))

print(f"\nNull counts for lag columns across full dataset:")
print(df_clean[['co2_lag1', 'co2_lag2', 'co2_lag3']].isnull().sum())
print("(Expected: 1× per country for lag1, 2× for lag2, 3× for lag3)")

In [ ]:
# 2.3a Verify co2_per_capita ──────────────────────────────

verification_countries = ['China', 'United States', 'India']
verification_years     = [2000, 2010, 2020]

verify_df = (
    df_clean[
        df_clean['country'].isin(verification_countries) &
        df_clean['year'].isin(verification_years)
    ]
    [['country', 'year', 'co2', 'population', 'co2_per_capita']]
    .copy()
)

verify_df['manual_co2_per_capita'] = (verify_df['co2'] * 1e6) / verify_df['population']
verify_df['difference'] = (
    verify_df['co2_per_capita'] - verify_df['manual_co2_per_capita']
).abs().round(6)

print("co2_per_capita cross-check (difference should be ≈ 0.0000):")
print(verify_df.to_string(index=False))

max_diff = verify_df['difference'].max()
if max_diff < 0.001:
    print(f"\n✓  Maximum absolute difference = {max_diff:.6f}  →  column is correctly computed.")
else:
    print(f"\n⚠  Maximum absolute difference = {max_diff:.6f}  →  investigate discrepancy.")

In [ ]:
# ── 2.3b GHG Intensity column ────────────────────────────────────────────

valid_mask = (
    df_clean['total_ghg'].notna() &
    df_clean['gdp'].notna() &
    (df_clean['gdp'] > 0)
)

df_clean['ghg_intensity'] = np.where(
    valid_mask,
    df_clean['total_ghg'] / df_clean['gdp'],
    np.nan
)

total_rows   = len(df_clean)
valid_rows   = valid_mask.sum()
missing_rows = total_rows - valid_rows

print(f"GHG intensity computed for : {valid_rows:,} rows ({valid_rows/total_rows:.1%})")
print(f"Missing (NaN)              : {missing_rows:,} rows ({missing_rows/total_rows:.1%})")

missing_by_country = (
    df_clean[df_clean['ghg_intensity'].isna()]
    .groupby('country')
    .size()
    .sort_values(ascending=False)
    .head(10)
    .reset_index(name='missing_years')
)
print("\nTop 10 countries with most missing ghg_intensity rows:")
print(missing_by_country.to_string(index=False))
print("\nNote: Missing values arise where GDP data is unavailable in the OWID dataset,")
print("      which is most common for very small or data-scarce nations.")

In [ ]:
# ── 2.4 Growth Rate Features ──────────────────────────────────────────

df_clean['co2_yoy_change'] = (
    df_clean.groupby('country')['co2'].diff()
)

df_clean['co2_yoy_pct_change'] = (
    df_clean.groupby('country')['co2'].pct_change() * 100
)

sufficient_data_countries = (
    df_clean[df_clean['co2'].notna()]
    .groupby('country')['year']
    .count()
    [lambda s: s >= 10]
    .index
)

avg_pct_growth = (
    df_clean[df_clean['country'].isin(sufficient_data_countries)]
    .groupby('country')['co2_yoy_pct_change']
    .mean()
    .dropna()
    .sort_values(ascending=False)
    .head(5)
    .reset_index()
    .rename(columns={'co2_yoy_pct_change': 'avg_annual_pct_growth'})
)
avg_pct_growth['avg_annual_pct_growth'] = avg_pct_growth['avg_annual_pct_growth'].round(2)

print("Top 5 countries \u2014 highest average annual CO2 growth rate since 1990:")
print(avg_pct_growth.to_string(index=False))

In [ ]:
# ── Top 5 largest absolute CO2 reductions since 1990 ───────────────────────

co2_net_change = (
    df_clean[
        df_clean['co2'].notna() &
        df_clean['country'].isin(sufficient_data_countries)
    ]
    .groupby('country')
    .apply(lambda grp: grp.loc[grp['year'].idxmax(), 'co2'] - grp.loc[grp['year'].idxmin(), 'co2'])
    .dropna()
    .sort_values()
    .head(5)
    .reset_index()
    .rename(columns={0: 'net_co2_change_mt'})
)
co2_net_change['net_co2_change_mt'] = co2_net_change['net_co2_change_mt'].round(2)

print("Top 5 countries \u2014 largest absolute CO2 reductions (MtCO2) from 1990 to latest year:")
print(co2_net_change.to_string(index=False))

In [ ]:
# ── 2.5 Assemble and save the final modelling DataFrame ─────────────────────

PROJECT_COUNTRIES = [
    'China', 'United States', 'India', 'Russia', 'Japan',
    'Germany', 'South Africa', 'Canada', 'Brazil', 'United Kingdom'
]

FEATURE_COLUMNS = [
    'country', 'year',
    'co2', 'co2_per_capita',
    'co2_5yr_rolling_mean',
    'co2_lag1', 'co2_lag2', 'co2_lag3',
    'co2_yoy_pct_change',
    'ghg_intensity'
]

df_model = (
    df_clean[df_clean['country'].isin(PROJECT_COUNTRIES)][FEATURE_COLUMNS]
    .reset_index(drop=True)
)

print("Final modelling DataFrame \u2014 shape:", df_model.shape)
print(f"Countries  : {df_model['country'].nunique()} × {df_model['year'].nunique()} years each")
print()
print("Null value summary:")
null_summary = df_model.isnull().sum().reset_index()
null_summary.columns = ['column', 'null_count']
null_summary['null_pct'] = (null_summary['null_count'] / len(df_model) * 100).round(1)
print(null_summary.to_string(index=False))

In [ ]:
# Preview the final dataset
print("First 15 rows of ghg_features.csv:")
df_model.head(15)

In [ ]:
# Save to CSV
output_path = 'ghg_features.csv'
df_model.to_csv(output_path, index=False)
print(f"Saved: {output_path}  ({len(df_model)} rows × {len(df_model.columns)} columns)")

df_reload = pd.read_csv(output_path)
assert df_reload.shape == df_model.shape, "Shape mismatch after reload!"
print("✓  Reload check passed \u2014 file is intact and ready to commit to GitHub.")

In [ ]:
# ── Summary statistics for the final feature set ─────────────────────
print("Descriptive statistics for numeric feature columns:")
df_model.describe().round(3)

# Week 3: Baseline ML Models – Regression

### 3.1 Problem Framing

**Prediction task:** Given a feature vector X describing country C in year Y (its CO₂ per-capita, 5-year rolling mean, lagged CO₂ values, year-on-year growth rate, and GHG intensity), predict that country's CO₂ emissions in year Y+1.

**Target variable:** `co2` — specifically `co2_next_year`, built by shifting each country's `co2` column back by one row so the label for year Y is the actual CO₂ value recorded in year Y+1.

**Input features:** `co2_per_capita`, `co2_5yr_rolling_mean`, `co2_lag1`, `co2_lag2`, `co2_lag3`, `co2_yoy_pct_change`, `ghg_intensity`, and `years_since_1990` — all carried over from Week 2's `df_model`.

**Why a supervised regression approach:** CO₂ emissions are a continuous numeric quantity, and historical labeled examples exist (year Y's features → year Y+1's actual emissions). Regression maps a country's current profile directly to a predicted numeric value and preserves magnitude of error — being off by 5 Mt vs 500 Mt should be scored very differently, which classification would not capture.

In [ ]:
# ── Build modelling frame: add years_since_1990 and the next-year target ─────

model_df = df_model.copy()
model_df['years_since_1990'] = model_df['year'] - 1990
model_df['co2_next_year'] = model_df.groupby('country')['co2'].shift(-1)
model_df = model_df.dropna(subset=['co2_next_year']).reset_index(drop=True)

FEATURE_COLS = ['co2_per_capita', 'co2_5yr_rolling_mean', 'co2_lag1', 'co2_lag2',
                'co2_lag3', 'co2_yoy_pct_change', 'ghg_intensity', 'years_since_1990']
TARGET_COL = 'co2_next_year'

print(f"Countries: {model_df['country'].nunique()} | Rows for modelling: {len(model_df)}")
print("\nPercent missing per feature:")
print((model_df[FEATURE_COLS].isna().mean() * 100).round(2))

### 3.2 Train-Test Split — Temporal, Not Random

Data is split by year, not shuffled randomly: **1990–2018 for training, 2019–2023 for testing.** Random splitting would let the model train on rows that come *after* some test rows chronologically — effectively letting it "see the future" during training, which inflates apparent accuracy and misrepresents how the model would actually be used (forecasting years that haven't happened from years that have). The 2019–2023 test window also deliberately includes the COVID-19 emissions dip (2020) and the recovery afterward — a real-world stress test of whether the model still performs reasonably when the future doesn't look like a simple continuation of the past.

In [ ]:
train_df = model_df[model_df['year'].between(1990, 2018)].copy()
test_df  = model_df[model_df['year'].between(2019, 2023)].copy()

split_summary = pd.DataFrame({
    'Country': PROJECT_COUNTRIES,
    'Train Samples': [len(train_df[train_df['country'] == c]) for c in PROJECT_COUNTRIES],
    'Test Samples':  [len(test_df[test_df['country'] == c]) for c in PROJECT_COUNTRIES],
})
split_summary

### 3.3 Naive Baseline Model

Predicted CO₂ for year Y+1 = actual CO₂ in year Y (no-change assumption). Any model built afterward should beat this baseline to be considered useful.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

plot_countries = PROJECT_COUNTRIES[:3]  # swap in specific country names if you prefer
baseline_results = []

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for country in PROJECT_COUNTRIES:
    test_c = test_df[test_df['country'] == country].dropna(subset=['co2', TARGET_COL]).sort_values('year')
    if len(test_c) == 0:
        continue
    mae = mean_absolute_error(test_c[TARGET_COL], test_c['co2'])
    rmse = np.sqrt(mean_squared_error(test_c[TARGET_COL], test_c['co2']))
    baseline_results.append({'Country': country, 'Baseline MAE': mae, 'Baseline RMSE': rmse})

    if country in plot_countries:
        ax = axes[plot_countries.index(country)]
        ax.plot(test_c['year'] + 1, test_c[TARGET_COL], marker='o', label='Actual')
        ax.plot(test_c['year'] + 1, test_c['co2'], marker='x', linestyle='--', label='Baseline Prediction')
        ax.set_title(f'{country}: Naive Baseline')
        ax.set_xlabel('Year'); ax.set_ylabel('CO2 (Mt)'); ax.legend()

plt.tight_layout()
plt.show()

baseline_results_df = pd.DataFrame(baseline_results)
baseline_results_df

### 3.4 Linear Regression

A separate model is trained per country (rather than one pooled model across all 10) so each country's own emissions scale and dynamics are captured directly, instead of being averaged across countries with very different absolute emission levels.

In [ ]:
from sklearn.linear_model import LinearRegression

lr_results, lr_models = [], {}
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for country in PROJECT_COUNTRIES:
    train_c = train_df[train_df['country'] == country].dropna(subset=FEATURE_COLS + [TARGET_COL])
    test_c  = test_df[test_df['country'] == country].dropna(subset=FEATURE_COLS + [TARGET_COL])
    if len(train_c) < 5 or len(test_c) == 0:
        continue

    model = LinearRegression().fit(train_c[FEATURE_COLS], train_c[TARGET_COL])
    y_pred = model.predict(test_c[FEATURE_COLS])

    lr_results.append({
        'Country': country,
        'LR MAE': mean_absolute_error(test_c[TARGET_COL], y_pred),
        'LR RMSE': np.sqrt(mean_squared_error(test_c[TARGET_COL], y_pred)),
    })
    lr_models[country] = model

    if country in plot_countries:
        ax = axes[plot_countries.index(country)]
        ax.plot(test_c['year'] + 1, test_c[TARGET_COL], marker='o', label='Actual')
        ax.plot(test_c['year'] + 1, y_pred, marker='s', linestyle='--', label='LR Prediction')
        ax.set_title(f'{country}: Linear Regression')
        ax.set_xlabel('Year'); ax.set_ylabel('CO2 (Mt)'); ax.legend()

plt.tight_layout()
plt.show()

lr_results_df = pd.DataFrame(lr_results)
coef_table = pd.DataFrame({c: m.coef_ for c, m in lr_models.items()}, index=FEATURE_COLS).T
display(lr_results_df)
display(coef_table)

**Interpreting the coefficients — fill this in after running the cell above:** look at `coef_table` and, for at least 2–3 specific countries, name the feature(s) with the largest absolute coefficient. In emissions data this is often `co2_lag1` or `co2_5yr_rolling_mean`, since next year's value is usually close to a smoothed version of recent history — but confirm this against your actual numbers rather than assuming it, since coefficient scale depends on each feature's own units and range.

### 3.5 Random Forest Regressor

Random Forest can capture non-linear relationships and interactions between features that Linear Regression cannot, at the cost of being less directly interpretable — hence the feature importance chart in place of coefficients.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf_results, rf_models = [], {}

for country in PROJECT_COUNTRIES:
    train_c = train_df[train_df['country'] == country].dropna(subset=FEATURE_COLS + [TARGET_COL])
    test_c  = test_df[test_df['country'] == country].dropna(subset=FEATURE_COLS + [TARGET_COL])
    if len(train_c) < 5 or len(test_c) == 0:
        continue

    model = RandomForestRegressor(n_estimators=100, random_state=42).fit(train_c[FEATURE_COLS], train_c[TARGET_COL])
    y_pred = model.predict(test_c[FEATURE_COLS])

    rf_results.append({
        'Country': country,
        'RF MAE': mean_absolute_error(test_c[TARGET_COL], y_pred),
        'RF RMSE': np.sqrt(mean_squared_error(test_c[TARGET_COL], y_pred)),
    })
    rf_models[country] = model

rf_results_df = pd.DataFrame(rf_results)
display(rf_results_df)

example_country = plot_countries[0]
if example_country in rf_models:
    importances = pd.Series(rf_models[example_country].feature_importances_, index=FEATURE_COLS).sort_values()
    plt.figure(figsize=(8, 5))
    importances.plot(kind='barh')
    plt.title(f'Random Forest Feature Importance — {example_country}')
    plt.xlabel('Importance')
    plt.tight_layout()
    plt.show()

### 3.6 Model Comparison

In [ ]:
comparison_df = baseline_results_df.merge(lr_results_df, on='Country').merge(rf_results_df, on='Country')

comparison_df['Best Model'] = comparison_df.apply(
    lambda r: min({'Baseline': r['Baseline MAE'], 'LR': r['LR MAE'], 'RF': r['RF MAE']},
                  key=lambda k: r[{'Baseline':'Baseline MAE','LR':'LR MAE','RF':'RF MAE'}[k]]),
    axis=1
)
comparison_df = comparison_df[['Country', 'Baseline MAE', 'LR MAE', 'RF MAE', 'Baseline RMSE', 'LR RMSE', 'RF RMSE', 'Best Model']]
comparison_df

**Conclusion — fill this in after running the cell above, based on your actual `comparison_df`:**
1. How many of the 10 countries had Baseline vs LR vs RF as the best model?
2. Did either trained model consistently beat the naive baseline, or did the baseline hold up surprisingly well (common in emissions data, since year-to-year change is often small relative to the noise a small model can introduce)?
3. Name 1–2 countries where the gap between best and worst model is largest, and connect that to what you saw in the coefficient table / feature importance chart above.

# Week 4: ETS(A,Ad,N) Forecasting with Confidence Intervals

**v3 change note:** ARIMA was replaced by ETS(A,Ad,N) for this project. The damping parameter prevents unbounded trend extrapolation over a 20-year horizon — more realistic for emissions data, particularly for countries with documented slowdowns (UK, Germany). The implementation is also simpler, requiring no stationarity testing or order selection.

In [ ]:
import matplotlib.pyplot as plt
from statsmodels.tsa.holtwinters import ExponentialSmoothing
import warnings
warnings.filterwarnings('ignore')

# Reuse df_model, PROJECT_COUNTRIES, train_df, test_df already built in Week 2/3 above
print(f"Countries: {len(PROJECT_COUNTRIES)} | Train rows: {len(train_df)} | Test rows: {len(test_df)}")

### 4.1 Concept Introduction — the ETS(A,Ad,N) Framework

ETS decomposes a time series into three components, each with an explicit specification here:

- **E (Error) — Additive:** the model's one-step-ahead residuals are added directly to the state at each time step (rather than multiplied), which suits emissions data where the scale of noise doesn't grow proportionally with the level in a way that requires a multiplicative form.
- **T (Trend) — Additive, Damped:** the trend component is added to the level, but decays toward zero over the forecast horizon via a damping parameter φ (0 < φ < 1). Instead of projecting a constant slope forever, the incremental change shrinks each period by a factor of φ, so the trend's influence fades out rather than compounding indefinitely.
- **S (Seasonality) — None:** this is annual data, one observation per country per year, so there is no within-year seasonal cycle to model.

**Why ETS(A,Ad,N) fits this problem:**
- Annual data has no seasonal pattern to capture, so the seasonal component is correctly omitted.
- The damped trend prevents unbounded long-range projections — unlike a unit-root ARIMA model, which can extrapolate a fixed trend forever, ETS(A,Ad,N) lets growth or decline taper off, which matches how real economies actually behave over multi-decade horizons.
- With only about 30 annual observations per country, a model needs to be parsimonious. ETS(A,Ad,N) has few free parameters (α, β*, φ, plus initial level/trend) compared to alternatives like a full ARIMA grid search, so it's less prone to overfitting a short series.
- It's physically sensible for emissions specifically: real economies tend to slow, plateau, or gradually reverse their emissions growth (due to policy, saturation of industrial capacity, or energy transition) — not grow at a constant compounding rate forever, which is what an undamped trend would imply.

In [ ]:
# ── 4.2 Model Fitting ────────────────────────────────────────────────

ets_fits = {}
ets_params = []

for country in PROJECT_COUNTRIES:
    train_co2 = (
        train_df[train_df['country'] == country]
        .sort_values('year')
        .set_index('year')['co2']
        .dropna()
    )

    model = ExponentialSmoothing(train_co2, trend='add', damped_trend=True, seasonal=None)
    fit = model.fit(optimized=True)
    ets_fits[country] = fit

    ets_params.append({
        'Country': country,
        'alpha (level)': round(fit.params['smoothing_level'], 4),
        'beta* (trend)': round(fit.params['smoothing_trend'], 4),
        'phi (damping)': round(fit.params['damping_trend'], 4),
    })

ets_params_df = pd.DataFrame(ets_params)
ets_params_df

In [ ]:
print("Fitted ETS(A,Ad,N) parameters for 3 example countries:\n")
for country in PROJECT_COUNTRIES[:3]:
    p = ets_fits[country].params
    print(f"{country}:")
    print(f"  alpha (smoothing level)  = {p['smoothing_level']:.4f}")
    print(f"  beta*  (smoothing trend) = {p['smoothing_trend']:.4f}")
    print(f"  phi    (damping trend)   = {p['damping_trend']:.4f}\n")

**Interpreting φ (phi) — fill this in against your actual `ets_params_df`:**

- **φ close to 1** (e.g. ≈ 0.99+) means almost no damping — the model expects the current trend to persist at nearly full strength for most of the 20-year horizon. This produces the steepest long-range forecasts and is worth sanity-checking against real-world plausibility.
- **φ noticeably lower** (e.g. ≈ 0.8) means the trend fades out faster — growth or decline flattens toward a stable level within a shorter window, which is often more realistic for a maturing economy or a country under active climate policy.
- **φ sitting exactly at an optimizer boundary** (very close to 0 or 1) can indicate the optimizer didn't find a clean interior solution — flag this rather than treating it as a confident estimate.

**Name your specific highest- and lowest-φ countries here** and state what each implies for that country's emissions trajectory.

### 4.3 Forecasting to 2043

In [ ]:
# ── Forecast helper: tries fit.get_forecast() first, falls back to simulation-based CI ──
# (statsmodels' HoltWintersResults doesn't always expose get_forecast/conf_int —
#  simulate() is the reliable cross-version approach for interval estimates)

def forecast_with_ci(fit, steps, start_year, n_sims=1000, alpha=0.05, seed=42):
    idx = pd.RangeIndex(start_year, start_year + steps)
    try:
        fc_result = fit.get_forecast(steps)
        mean_fc = pd.Series(fc_result.predicted_mean.values, index=idx)
        ci = fc_result.conf_int(alpha=alpha)
        lower = pd.Series(ci.iloc[:, 0].values, index=idx)
        upper = pd.Series(ci.iloc[:, 1].values, index=idx)
    except Exception:
        mean_fc = pd.Series(fit.forecast(steps).values, index=idx)
        sims = fit.simulate(steps, repetitions=n_sims, error='add', random_state=seed)
        sims.index = idx
        lower = sims.quantile(alpha / 2, axis=1)
        upper = sims.quantile(1 - alpha / 2, axis=1)
    return mean_fc, lower, upper


FORECAST_START = 2019
FORECAST_END = 2043
FORECAST_STEPS = FORECAST_END - FORECAST_START + 1  # 25 steps: 2019..2043

forecast_results = {}
for country in PROJECT_COUNTRIES:
    fit = ets_fits[country]
    mean_fc, lower, upper = forecast_with_ci(fit, FORECAST_STEPS, FORECAST_START)
    forecast_results[country] = {'mean': mean_fc, 'lower': lower, 'upper': upper}

print(f"Forecasts generated for {FORECAST_STEPS} years ({FORECAST_START}–{FORECAST_END}) across {len(forecast_results)} countries.")

In [ ]:
# ── Forecast plots — one panel per country, with actuals / fitted / holdout / forecast + CI ──

fig, axes = plt.subplots(5, 2, figsize=(16, 22))
axes = axes.flatten()

for i, country in enumerate(PROJECT_COUNTRIES):
    ax = axes[i]
    train_co2 = train_df[train_df['country'] == country].sort_values('year').set_index('year')['co2']
    test_co2  = test_df[test_df['country'] == country].sort_values('year').set_index('year')['co2']
    fit = ets_fits[country]
    fc = forecast_results[country]

    ax.plot(train_co2.index, train_co2.values, color='#1d3557', linewidth=1.8, label='Historical Actual (1990–2018)')
    ax.plot(train_co2.index, fit.fittedvalues.values, color='#e9c46a', linewidth=1.3, linestyle='--', label='Fitted (train)')
    ax.plot(test_co2.index, test_co2.values, color='#2a9d8f', linewidth=1.8, marker='o', markersize=4, label='Holdout Actual (2019–2023)')
    ax.plot(fc['mean'].index, fc['mean'].values, color='#e63946', linewidth=1.8, label='Forecast (2019–2043)')
    ax.fill_between(fc['mean'].index, fc['lower'].values, fc['upper'].values, color='#e63946', alpha=0.15, label='95% CI')

    ax.set_title(country, fontsize=12, fontweight='bold')
    ax.set_xlabel('Year')
    ax.set_ylabel('CO₂ (Mt)')
    ax.grid(alpha=0.25)
    if i == 0:
        ax.legend(fontsize=8, loc='upper left')

plt.suptitle('ETS(A,Ad,N) Forecasts: Actuals, Fitted, Holdout, and 2043 Projection with 95% CI', fontsize=15, fontweight='bold', y=1.005)
plt.tight_layout()
plt.show()

### 4.4 Trend Interpretation

**For at least 3 countries, fill in based on your actual forecast plots and `ets_params_df` above:**

- **United Kingdom:** does the damped projection show emissions flattening or continuing to fall toward 2043? The UK has legislated a net-zero-by-2050 target and a documented multi-decade decline in coal use — note whether the forecast's trajectory (and its φ) is consistent with that policy backdrop, or whether the damping looks too aggressive/too mild relative to known reductions.
- **Germany:** check whether the fit converged to an interior φ or landed on an optimizer boundary (flagged in Week 4's parameter table) — if boundary, treat the shape of this forecast with more caution than the others.
- **India:** does the forecast show continued growth tapering only gradually, consistent with an economy still industrializing? Compare the shape against China's, which has stated a peak-emissions target for the mid-2020s — does China's damped curve show growth topping out sooner than India's?
- **CI width over the horizon:** report whether the shaded band visibly widens from 2024 to 2043 for these countries. A widening CI is expected and appropriate — it reflects genuine growing uncertainty about a 20-year-ahead single-country forecast built from ~30 data points, and should be read as a caution against treating the central forecast line as a precise prediction that far out.

### 4.5 Forecast Summary Table

In [ ]:
summary_rows = []
for country in PROJECT_COUNTRIES:
    fc = forecast_results[country]['mean']
    actual_2020 = df_model[(df_model['country'] == country) & (df_model['year'] == 2020)]['co2']
    actual_2020_val = actual_2020.values[0] if len(actual_2020) else np.nan
    fc_2040 = fc.get(2040, np.nan)
    pct_change = ((fc_2040 - actual_2020_val) / actual_2020_val * 100) if pd.notna(actual_2020_val) and actual_2020_val != 0 else np.nan

    summary_rows.append({
        'Country': country,
        '2030 Forecast (MtCO2)': round(fc.get(2030, np.nan), 1),
        '2035 Forecast (MtCO2)': round(fc.get(2035, np.nan), 1),
        '2040 Forecast (MtCO2)': round(fc_2040, 1),
        '2020 Actual (MtCO2)': round(actual_2020_val, 1) if pd.notna(actual_2020_val) else np.nan,
        '% Change 2020→2040': round(pct_change, 1) if pd.notna(pct_change) else np.nan,
    })

forecast_summary_df = pd.DataFrame(summary_rows)
forecast_summary_df

### 4.6 Model Validation — Consolidated Four-Model Comparison

In [ ]:
# ── Recompute Week 3's three models here (self-contained notebook, separate kernel) ──

plot_countries = PROJECT_COUNTRIES[:3]

# Naive baseline
baseline_results = []
for country in PROJECT_COUNTRIES:
    test_c = test_df[test_df['country'] == country].dropna(subset=['co2', TARGET_COL]).sort_values('year')
    if len(test_c) == 0:
        continue
    baseline_results.append({
        'Country': country,
        'Baseline MAE': mean_absolute_error(test_c[TARGET_COL], test_c['co2']),
        'Baseline RMSE': np.sqrt(mean_squared_error(test_c[TARGET_COL], test_c['co2'])),
    })
baseline_results_df = pd.DataFrame(baseline_results)

# Linear Regression
lr_results = []
for country in PROJECT_COUNTRIES:
    train_c = train_df[train_df['country'] == country].dropna(subset=FEATURE_COLS + [TARGET_COL])
    test_c  = test_df[test_df['country'] == country].dropna(subset=FEATURE_COLS + [TARGET_COL])
    if len(train_c) < 5 or len(test_c) == 0:
        continue
    model = LinearRegression().fit(train_c[FEATURE_COLS], train_c[TARGET_COL])
    y_pred = model.predict(test_c[FEATURE_COLS])
    lr_results.append({
        'Country': country,
        'LR MAE': mean_absolute_error(test_c[TARGET_COL], y_pred),
        'LR RMSE': np.sqrt(mean_squared_error(test_c[TARGET_COL], y_pred)),
    })
lr_results_df = pd.DataFrame(lr_results)

# Random Forest
rf_results = []
for country in PROJECT_COUNTRIES:
    train_c = train_df[train_df['country'] == country].dropna(subset=FEATURE_COLS + [TARGET_COL])
    test_c  = test_df[test_df['country'] == country].dropna(subset=FEATURE_COLS + [TARGET_COL])
    if len(train_c) < 5 or len(test_c) == 0:
        continue
    model = RandomForestRegressor(n_estimators=100, random_state=42).fit(train_c[FEATURE_COLS], train_c[TARGET_COL])
    y_pred = model.predict(test_c[FEATURE_COLS])
    rf_results.append({
        'Country': country,
        'RF MAE': mean_absolute_error(test_c[TARGET_COL], y_pred),
        'RF RMSE': np.sqrt(mean_squared_error(test_c[TARGET_COL], y_pred)),
    })
rf_results_df = pd.DataFrame(rf_results)

# ETS — score against the 2019-2023 holdout portion of each country's forecast
ets_results = []
for country in PROJECT_COUNTRIES:
    test_co2 = test_df[test_df['country'] == country].sort_values('year').set_index('year')['co2'].dropna()
    if len(test_co2) == 0:
        continue
    fc_holdout = forecast_results[country]['mean'].reindex(test_co2.index)
    ets_results.append({
        'Country': country,
        'ETS MAE': mean_absolute_error(test_co2, fc_holdout),
        'ETS RMSE': np.sqrt(mean_squared_error(test_co2, fc_holdout)),
    })
ets_results_df = pd.DataFrame(ets_results)

# Consolidated four-model table
comparison_df = (
    baseline_results_df
    .merge(lr_results_df, on='Country')
    .merge(rf_results_df, on='Country')
    .merge(ets_results_df, on='Country')
)

def best_model(row):
    scores = {'Baseline': row['Baseline MAE'], 'LR': row['LR MAE'], 'RF': row['RF MAE'], 'ETS': row['ETS MAE']}
    return min(scores, key=scores.get)

comparison_df['Best Model'] = comparison_df.apply(best_model, axis=1)
comparison_df = comparison_df[[
    'Country', 'Baseline MAE', 'LR MAE', 'RF MAE', 'ETS MAE',
    'Baseline RMSE', 'LR RMSE', 'RF RMSE', 'ETS RMSE', 'Best Model'
]]
comparison_df

**Conclusion — fill this in after running the cell above, based on your actual `comparison_df`:**

1. Which model wins most often across the 10 countries — does ETS outperform the regression models on more countries, fewer, or roughly the same number?
2. Note the asymmetry in what's being compared: ETS forecasts multiple years ahead from a fixed 2018 origin using only its own past values, while Linear Regression and Random Forest predict one step ahead using refreshed features (including lags) at each test point — this makes the comparison informative but not perfectly apples-to-apples, and is worth stating explicitly rather than treating ETS's MAE as directly equivalent in difficulty.
3. Name 1–2 countries where ETS does notably better or worse than the regression models, and connect that to what you saw in the φ (damping) values — e.g. a country with reasonable damping producing a more stable multi-year forecast versus one where damping sat at a boundary and produced a less reliable projection.

# Week 5: Scenario Analysis (Optional)

_Note: this section reuses `df_model`, `PROJECT_COUNTRIES`, and `bau_forecast` built in the Week 4 section above — no data is reloaded._

### 5.1 Scenario Design

- **Scenario A — Business as Usual (BAU):** no policy change; uses the ETS(A,Ad,N) forecast from Week 4 as-is.
- **Scenario B — Moderate Mitigation:** applies a compounding 2% annual reduction to the BAU forecast, starting in 2025. Each year's value is 2% lower than what it would otherwise have been under BAU, compounding forward (i.e. `BAU_year × (1 − 0.02)^(year − 2024)`).
- **Scenario C — Aggressive Mitigation:** same compounding logic, but at a 5% annual reduction rate.

**Basis and limitations:** these are **illustrative, not scientifically calibrated** reduction pathways. Real mitigation trajectories depend on specific policy instruments (carbon pricing, energy mix transitions, efficiency standards), sectoral composition, and enforcement — none of which are modeled here. A flat percentage reduction applied uniformly to every country ignores that some countries have far more room to cut (e.g. coal-heavy grids) than others (e.g. already low-carbon grids), and it ignores diminishing returns or acceleration effects that real decarbonisation pathways typically show. Treat Scenarios B and C as a simple illustrative device for comparing *relative* outcomes across countries under a common assumption, not as country-specific policy forecasts.

### 5.2 Scenario Calculation

In [ ]:
SCENARIO_START = 2025
SCENARIO_END = 2040
REDUCTION_RATES = {'BAU': 0.0, 'Moderate Mitigation': 0.02, 'Aggressive Mitigation': 0.05}

scenario_rows = []
for country in PROJECT_COUNTRIES:
    fc = forecast_results[country]['mean']
    for year in range(SCENARIO_START, SCENARIO_END + 1):
        bau_value = fc.get(year, np.nan)
        t = year - (SCENARIO_START - 1)  # years since 2024 baseline (t=1 for 2025)
        for scenario, rate in REDUCTION_RATES.items():
            co2_projected = bau_value * ((1 - rate) ** t)
            scenario_rows.append({
                'country': country,
                'year': year,
                'scenario': scenario,
                'co2_projected': round(co2_projected, 2),
            })

scenario_projections_df = pd.DataFrame(scenario_rows)
print("Scenario projections shape:", scenario_projections_df.shape)
scenario_projections_df.head(9)

In [ ]:
scenario_projections_df.to_csv('scenario_projections.csv', index=False)
print("Saved: scenario_projections.csv "
      f"({len(scenario_projections_df)} rows × {len(scenario_projections_df.columns)} columns)")
print("Remember to commit this file to your GitHub repository alongside this notebook.")

### 5.3 Scenario Visualisations

In [ ]:
# ── Per-country overlay: historical actuals + 3 scenario lines + 1990 benchmark ──

scenario_colors = {'BAU': '#457B9D', 'Moderate Mitigation': '#F4A261', 'Aggressive Mitigation': '#2A9D8F'}

fig, axes = plt.subplots(5, 2, figsize=(16, 22))
axes = axes.flatten()

for i, country in enumerate(PROJECT_COUNTRIES):
    ax = axes[i]

    hist = df_model[(df_model['country'] == country) & (df_model['year'].between(1990, 2024))].sort_values('year')
    ax.plot(hist['year'], hist['co2'], color='grey', linewidth=1.5, label='Historical Actual (1990–2024)')

    fc = forecast_results[country]['mean']
    pre2025_years = [y for y in range(2020, 2025)]
    for scenario in REDUCTION_RATES:
        pre_vals = [fc.get(y, np.nan) for y in pre2025_years]  # scenarios haven't diverged yet before 2025
        scen_data = scenario_projections_df[
            (scenario_projections_df['country'] == country) & (scenario_projections_df['scenario'] == scenario)
        ].sort_values('year')
        plot_years = pre2025_years + scen_data['year'].tolist()
        plot_vals = pre_vals + scen_data['co2_projected'].tolist()
        ax.plot(plot_years, plot_vals, color=scenario_colors[scenario], linewidth=2, label=scenario)

    benchmark_1990 = df_model[(df_model['country'] == country) & (df_model['year'] == 1990)]['co2']
    if len(benchmark_1990):
        ax.axhline(benchmark_1990.values[0], color='black', linestyle=':', linewidth=1, alpha=0.6,
                   label='1990 Level (benchmark)')

    ax.set_title(country, fontsize=12, fontweight='bold')
    ax.set_xlabel('Year')
    ax.set_ylabel('CO2 (Mt)')
    ax.grid(alpha=0.25)
    if i == 0:
        ax.legend(fontsize=7, loc='upper left')

plt.suptitle('Scenario Analysis: BAU vs Moderate vs Aggressive Mitigation (2020–2040)', fontsize=15, fontweight='bold', y=1.005)
plt.tight_layout()
plt.show()

In [ ]:
# ── Global aggregate chart: sum of all 10 countries per scenario ──

global_agg = (
    scenario_projections_df
    .groupby(['scenario', 'year'])['co2_projected']
    .sum()
    .reset_index()
)

plt.figure(figsize=(11, 6))
for scenario in REDUCTION_RATES:
    subset = global_agg[global_agg['scenario'] == scenario].sort_values('year')
    plt.plot(subset['year'], subset['co2_projected'], color=scenario_colors[scenario], linewidth=2.5, label=scenario)

plt.title('Global Aggregate CO₂ Projection Across 3 Scenarios (2025–2040) — 10 Project Countries', fontsize=13, fontweight='bold')
plt.xlabel('Year')
plt.ylabel('Total CO₂ (Mt), summed across 10 countries')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 5.4 Impact Summary

In [ ]:
cumulative_emissions = (
    scenario_projections_df
    .groupby(['country', 'scenario'])['co2_projected']
    .sum()
    .reset_index()
    .rename(columns={'co2_projected': 'cumulative_co2_2025_2040'})
)

cumulative_pivot = cumulative_emissions.pivot(index='country', columns='scenario', values='cumulative_co2_2025_2040')
cumulative_pivot = cumulative_pivot[['BAU', 'Moderate Mitigation', 'Aggressive Mitigation']]

x = np.arange(len(cumulative_pivot))
width = 0.25

fig, ax = plt.subplots(figsize=(14, 6))
for j, scenario in enumerate(cumulative_pivot.columns):
    ax.bar(x + (j - 1) * width, cumulative_pivot[scenario], width, label=scenario, color=scenario_colors[scenario])

ax.set_xticks(x)
ax.set_xticklabels(cumulative_pivot.index, rotation=30, ha='right')
ax.set_ylabel('Cumulative CO₂, 2025–2040 (Mt)')
ax.set_title('Cumulative Emissions by Country Across 3 Scenarios (2025–2040)', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

cumulative_pivot['BAU minus Aggressive (Mt avoided)'] = cumulative_pivot['BAU'] - cumulative_pivot['Aggressive Mitigation']
cumulative_pivot.sort_values('BAU minus Aggressive (Mt avoided)', ascending=False)

**Interpretation — fill this in based on your actual `cumulative_pivot` output:**

Report which countries show the largest **absolute** avoided emissions (Mt) under Aggressive Mitigation versus BAU — these will typically be the countries with the highest BAU trajectory to begin with, since a flat 5% compounding cut removes more tonnage from a larger base. Separately, note whether the *ranking* changes if you instead look at **percentage** avoided rather than absolute Mt (a smaller emitter can show a large percentage benefit while still contributing little in absolute terms). State 2–3 sentences on which countries benefit most in absolute terms, which benefit most proportionally, and why those are not necessarily the same countries.